# cls_compare.ipynb
**Does:** Experiment 1 - builds the particle tables, then compares feature models and CNNs on the particles the pipeline finds, plus the resolution test (Part H).
**Uses:** `src/scale.py`, `src/hybrid_segment.py`, `src/piv_compare.py`, `src/cls_compare.py` (run piv_compare.ipynb Part 0 first).
**Outputs:** `cls_tables/` (particle tables, reused by cls_compare2.ipynb) and `cls_outputs/<n>/` (scores, decisions, test results, figures).

## Settings

In [ ]:
# settings for this run (GPU with High-RAM needed for Part C)
DATA_NAME, RARE_NAME = 'run_13um', 'run_13um_rare'
RUN_ID      = None               # None = new output folder; a number = continue that run
CLS_DATA    = 'cls_tables'       # particle tables (shared by both classification notebooks)
SOURCE      = 'pipeline'         # 'pipeline' = hybrid_segment + tracking; 'truth' = perfect detections
QF_FILE     = 'hybrid_segment_outputs/quality_filter.joblib'
QF_TAU, QF_TRAIN_STEP = 0.3, 2
MOTION, TRACKER = 'ransac', 'tracking'
PIV_TRACKS  = 'piv_outputs/0/tracks'   # reuse the val/test tracks of a piv_compare run (None = track here)
EVERY_TRAIN = 3                  # every 3rd frame of the training sequences
SCHEME      = 'amaral7u'         # the 8 classes
SCHEMES     = ['amaral7u', 'amaral7', 'song6u', 'durkin9u']
FEATURE_MODELS = None            # None = all
CNN_MODELS  = None               # None = all
CNN_SEEDS   = {'resnet18_amaral': (0,), 'mobilenet_w': (0,), 'native_cnn': (0, 1, 2), 'hybrid': (0, 1, 2)}
CNN_EPOCHS, CNN_PATIENCE = 30, 6
CLASS_POWER = 0.5                # class balancing: weight = (largest class / class)^power
SELECT_BY   = 'acc_bal_hmean'    # chosen on validation
K_CHOICES   = (1, 3, 10, None)   # detections per particle combined (None = whole track)
TAU_CHOICES = (0.0, 0.25, 0.5, 0.75, 1.0)   # logit adjustment strength
REJECT      = 0.5                # false detection only if its probability >= this
MAX_ENSEMBLE = 3
CARBON_SCALES = 'val'
CARBON_TAU  = 0.0
SIZE_EDGES_UM = (200, 300, 500)  # Part H size bins (um)
SIZE_EDGES_PX = (9, 14, 23)      # Part H size bins (px)

## Setup

In [ ]:
# mount Drive, load the scripts, set the pixel scale and create the output folder
import os, sys, glob, json, shutil, time
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from google.colab import drive
from IPython.display import display
pd.set_option('display.max_columns', None, 'display.width', 250)

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/snocam_pipeline'
DATA, RARE = f'/content/{DATA_NAME}', f'/content/{RARE_NAME}'
sh = get_ipython().system
sh('pip install -q timm lightgbm xgboost openpyxl openpiv')
shutil.copytree(f'{DRIVE}/src', '/content/src', dirs_exist_ok=True, ignore=shutil.ignore_patterns('__pycache__'))
sys.path.append('/content/src')


def restore(name):
    # copy a dataset or table folder from Drive to /content (from its .tar if there is one)
    if not os.path.exists(f'/content/{name}'):
        if os.path.exists(f'{DRIVE}/{name}.tar'):
            sh(f'tar -xf "{DRIVE}/{name}.tar" -C /content')
        elif os.path.exists(f'{DRIVE}/{name}'):
            shutil.copytree(f'{DRIVE}/{name}', f'/content/{name}')
    return os.path.exists(f'/content/{name}')


import scale as SC
CLS = f'/content/{CLS_DATA}'
restore(CLS_DATA)
RES_FILE, DONE_FILE = f'{CLS}/resolution.json', f'{CLS}/tables_complete.json'
if os.path.exists(RES_FILE):            # tables already built: use their pixel scale
    r = json.load(open(RES_FILE))
    res = SC.configure(r['um_per_px'], min_area=r['min_area_px'], min_area_mode=r['min_area_mode'],
                       canvas=r['canvas_px'], source=RES_FILE)
else:                                   # otherwise read it from the data
    assert restore(DATA_NAME), f'no {DATA_NAME} in {DRIVE}: run pipeline.ipynb Part A first'
    res = SC.configure_from_data(DATA)
print('resolution:', res)
import torch
GPU = torch.cuda.is_available()

import cls_compare as cc
FEATURE_MODELS = FEATURE_MODELS or list(cc.FEATURE_MODELS)
CNN_MODELS = CNN_MODELS or list(cc.CNN_MODELS)
os.makedirs(f'{DRIVE}/cls_outputs', exist_ok=True)
if RUN_ID is None:
    used = [int(n) for n in os.listdir(f'{DRIVE}/cls_outputs') if n.isdigit()]
    RUN_ID = max(used) + 1 if used else 0
OUT = f'{DRIVE}/cls_outputs/{RUN_ID}'
for sub in ('probs', 'models'):
    os.makedirs(f'{OUT}/{sub}', exist_ok=True)
SETTINGS = {k: v for k, v in globals().items() if k.isupper() and isinstance(v, (int, float, str, list, tuple, dict, type(None)))}
json.dump(dict(SETTINGS, resolution=res), open(f'{OUT}/settings.json', 'w'), indent=1, default=str)
print('outputs:', OUT, '| GPU:', GPU)


def backup(name):
    sh(f'mkdir -p "{DRIVE}/{name}" && rsync -a "/content/{name}/" "{DRIVE}/{name}/"')


def snapshot(name):
    sh(f'tar -cf "/content/{name}.tar" -C /content {name} && mv "/content/{name}.tar" "{DRIVE}/{name}.tar" && rm -rf "{DRIVE}/{name}"')


def all_sequences():
    return sorted(glob.glob(f'{DATA}/seq*_*')) + (sorted(glob.glob(f'{RARE}/rare*_train')) if os.path.exists(RARE) else [])


def probs_path(model, split):
    return f'{OUT}/probs/{model}_{split}.npy'

## Status

In [ ]:
# what is already built and scored
DONE_FILE = f'{CLS}/tables_complete.json'      # written at the end of Part A
built = [os.path.basename(f)[:-4] for f in glob.glob(f'{CLS}/*.pkl')]
if os.path.exists(DONE_FILE):
    print(f'particle tables complete in {CLS_DATA}: {len(built)} sequences (the synthetic data are not needed any more)')
else:
    restore(DATA_NAME)
    if RARE:
        restore(RARE_NAME)
    seqs = all_sequences()
    by = pd.Series([s.split('_')[-1] for s in seqs]).value_counts().to_dict()
    print(f'sequences: {by}')
    print(f'particle tables built in {CLS_DATA}: {len(built)}/{len(seqs)} -> run Part A')
    print('quality filter already trained:', os.path.exists(f'{DRIVE}/{QF_FILE}'))
done = sorted({os.path.basename(f)[:-8] for f in glob.glob(f"{OUT}/probs/*_val.npy")})
print('models already scored in this run:', done)
print('final configuration saved:', os.path.exists(f'{OUT}/final_config.json'))

## Helpers

In [ ]:
# helper functions used by every Part (run once per session)
COLS = ['accuracy', 'balanced_accuracy', 'acc_bal_hmean', 'flux_balanced_accuracy', 'macro_f1', 'carbon_weighted_accuracy',
        'accuracy_esd_ge_500um', 'carbon_error_found', 'carbon_error_all', 'n_particles']
_cache = {}

def tables(split, crops=False):
    # the particle table of one split (labels under SCHEME) and, if asked, its crops; cached
    if (split, crops) not in _cache:
        T, C = cc.load_tables(CLS, (split,), crops=crops)
        _cache[(split, crops)] = (cc.add_labels(T, SCHEME), C)
    return _cache[(split, crops)]

def truth_arr():
    if 'truth_arr' not in _cache:
        _cache['truth_arr'] = cc.truth_arrivals(CLS, ('val', 'test'))
    return _cache['truth_arr']

def prior():
    if 'prior' not in _cache:
        _cache['prior'] = cc.natural_prior(tables('train')[0])
    return _cache['prior']

def classes_of(name):
    return np.array(json.load(open(f'{OUT}/probs/{name}_classes.json')))

def model_probs(members, split):
    # class probabilities per detection for one model, or the geometric mean of several (an ensemble)
    if isinstance(members, str):
        members = [members]
    if len(members) == 1:
        return np.load(probs_path(members[0], split)), classes_of(members[0])
    cl = np.array(sorted(set().union(*[set(classes_of(m)) for m in members])))
    L = 0
    for m in members:
        P = np.full((len(tables(split)[0]), len(cl)), 1e-6)
        P[:, np.searchsorted(cl, classes_of(m))] = np.load(probs_path(m, split))
        L = L + np.log(P / P.sum(1, keepdims=True))
    L = L / len(members)
    P = np.exp(L - L.max(1, keepdims=True))
    return P / P.sum(1, keepdims=True), cl

def evaluate(members, split='val', K=None, how='geo', tau=0.0, size_model=None, scales=None):
    P, cl = model_probs(members, split)
    return cc.evaluate_probs(tables(split)[0], P, cl, K, how, REJECT, tau, prior(), size_model, scales, truth_arr(), SCHEME,
                             carbon_tau=CARBON_TAU)

def run_config(c, split, carbon=None):
    carbon = carbon or {}
    return evaluate(c['members'], split, c['K'], c['how'], c['tau'], carbon.get('size_model'), carbon.get('scales'))

def show(df, sort=None):
    d = df.copy()
    pc_cols = [c for c in COLS if c in d.columns and c != 'n_particles']
    d[pc_cols] = (d[pc_cols].astype(float) * 100).round(1)
    display(d.sort_values(sort, ascending=False) if sort else d)
print('helpers ready')

## Part A: particle tables

In [ ]:
# build the particle tables: detections, tracks, 81 features, an 80 px crop and the truth of every detection
if os.path.exists(DONE_FILE):
    print('particle tables complete')
else:
    restore(DATA_NAME)
    if RARE:
        restore(RARE_NAME)
    if SOURCE == 'pipeline':
        import piv_compare as pc
        pc.use_hybrid_segment(f'{DRIVE}/{QF_FILE}', QF_TAU, data_dir=DATA, frame_step=QF_TRAIN_STEP)
    os.makedirs(CLS, exist_ok=True)
    json.dump(SC.summary(), open(RES_FILE, 'w'), indent=1)
    tracks_dir = f'{DRIVE}/{PIV_TRACKS}' if PIV_TRACKS else None
    seqs = all_sequences()
    for d in seqs:
        sid = os.path.basename(d)
        if os.path.exists(f'{CLS}/{sid}.pkl'):
            continue
        is_train = sid.endswith('_train')
        tcsv = f'{tracks_dir}/{sid}/{MOTION}__{TRACKER}.csv' if tracks_dir else None
        cc.build_sequence(d, CLS, SOURCE, EVERY_TRAIN if is_train else 1, track=not is_train, tracks_csv=tcsv,
                          motion=MOTION, tracker=TRACKER)
        backup(CLS_DATA)
    json.dump(dict(sequences=[os.path.basename(d) for d in seqs], resolution=SC.summary()), open(DONE_FILE, 'w'), indent=1)
    snapshot(CLS_DATA)                  # complete -> one archive in Drive for fast restores

In [ ]:
# particles and distinct sprites per class, false detections and the size bias of the outlines
T = pd.concat([tables(s)[0] for s in ('train', 'val', 'test')], ignore_index=True)
p = T.drop_duplicates('pkey')
tab = pd.concat({'particles': p.groupby(['label', 'split']).size().unstack(fill_value=0),
                 'sprites': p[p.uid > 0].groupby(['label', 'split']).source_uid.nunique().unstack(fill_value=0)}, axis=1)
display(tab)
print('false detections, % of detections:', T.groupby('split').uid.apply(lambda u: round(100 * (u == 0).mean(), 1)).to_dict())
q = p[p.uid > 0]
print('median measured / true ESD:', (q.esd_um / q.esd_true).groupby(q.split).median().round(2).to_dict())
del T, p, q

## Part B: feature models

In [ ]:
# train every feature model once and score it on validation
import joblib
tr, Tv, Tt = tables('train')[0], tables('val')[0], tables('test')[0]
for name in FEATURE_MODELS:
    if os.path.exists(probs_path(name, 'val')):
        continue
    t0 = time.time()
    m = cc.fit_feature_model(name, tr, power=CLASS_POWER)
    for split, T_ in (('val', Tv), ('test', Tt), ('train', tr)):          # train = in-sample, for Part E
        np.save(probs_path(name, split), cc.predict_feature_model(m, T_))
    json.dump(list(m.labels_), open(f'{OUT}/probs/{name}_classes.json', 'w'))
    joblib.dump(m, f'{OUT}/models/{name}.joblib', compress=3)
    print(f'{name}: {time.time() - t0:.0f} s', flush=True)
if not os.path.exists(probs_path('always_aggregate', 'val')):
    cl = np.array(sorted(tr.label.unique()))
    for split, T_ in (('val', Tv), ('test', Tt)):
        np.save(probs_path('always_aggregate', split), cc.constant_probs(len(T_), cl))
    json.dump(list(cl), open(f'{OUT}/probs/always_aggregate_classes.json', 'w'))

B = pd.DataFrame({n: evaluate(n)[0] for n in ['always_aggregate'] + FEATURE_MODELS}).T
B[COLS].to_csv(f'{OUT}/validation_feature_models.csv')
show(B[COLS], SELECT_BY)

In [ ]:
# the other class schemes with the best feature model (validation)
B = pd.read_csv(f'{OUT}/validation_feature_models.csv', index_col=0).drop('always_aggregate', errors='ignore')
best_feat = B[SELECT_BY].idxmax()
if not os.path.exists(f'{OUT}/validation_schemes.csv'):
    rows = []
    for sch in SCHEMES:
        tr_s, Tv_s = cc.add_labels(tables('train')[0], sch), cc.add_labels(tables('val')[0], sch)
        m = cc.fit_feature_model(best_feat, tr_s, power=CLASS_POWER)
        r, *_ = cc.evaluate_probs(Tv_s, cc.predict_feature_model(m, Tv_s), m.labels_, None, 'geo', REJECT, 0, None,
                                  None, None, truth_arr(), sch)
        rows.append(dict(scheme=sch, model=best_feat, n_classes=len(m.labels_), **r))
    pd.DataFrame(rows).set_index('scheme').to_csv(f'{OUT}/validation_schemes.csv')
show(pd.read_csv(f'{OUT}/validation_schemes.csv', index_col=0))

## Part C: CNNs (GPU)

In [ ]:
# train every CNN on the crops and score it on validation
assert GPU, 'Part C needs a GPU: Runtime -> Change runtime type -> T4 GPU'
trC, Ctr = tables('train', crops=True)
Tv, Cv = tables('val', crops=True)
Tt, Ct = tables('test', crops=True)
print('crops in memory:', Ctr.shape, Cv.shape, Ct.shape)

def finish(arch, seeds):
    # the model = geometric mean of its seeds
    if os.path.exists(probs_path(arch, 'val')) or not all(os.path.exists(probs_path(f'{arch}_s{s}', 'val')) for s in seeds):
        return
    for split in ('val', 'test', 'train'):
        L = np.mean([np.log(np.load(probs_path(f'{arch}_s{s}', split))) for s in seeds], 0)
        P = np.exp(L - L.max(1, keepdims=True)); np.save(probs_path(arch, split), P / P.sum(1, keepdims=True))
    shutil.copy(f'{OUT}/probs/{arch}_s{seeds[0]}_classes.json', f'{OUT}/probs/{arch}_classes.json')
    shutil.copy(f'{OUT}/probs/{arch}_s{seeds[0]}_train_rows.npy', f'{OUT}/probs/{arch}_train_rows.npy')

sub = np.sort(np.random.default_rng(0).choice(len(trC), min(30000, len(trC)), replace=False))   # in-sample rows (Part E)
for arch in CNN_MODELS:
    seeds = CNN_SEEDS.get(arch, (0,))
    for seed in seeds:
        tag = f'{arch}_s{seed}'
        if os.path.exists(probs_path(tag, 'val')):
            continue
        t0 = time.time()
        net = cc.train_cnn(arch, trC, Ctr, Tv, Cv, epochs=CNN_EPOCHS, patience=CNN_PATIENCE, power=CLASS_POWER, seed=seed,
                           log=lambda r: print(tag, r, flush=True))
        cc.save_cnn(net, f'{OUT}/models/{tag}.pt')
        net.history_.to_csv(f'{OUT}/models/{tag}_history.csv', index=False)
        np.save(probs_path(tag, 'test'), cc.predict_cnn(net, Tt, Ct))
        np.save(probs_path(tag, 'train'), cc.predict_cnn(net, trC.iloc[sub], Ctr[sub], tta=False))
        np.save(f'{OUT}/probs/{tag}_train_rows.npy', sub)
        json.dump(list(net.classes_), open(f'{OUT}/probs/{tag}_classes.json', 'w'))
        np.save(probs_path(tag, 'val'), cc.predict_cnn(net, Tv, Cv))          # written last: marks this seed done
        print(f'{tag}: {time.time() - t0:.0f} s, best epoch {int(net.history_.val_score.idxmax())}', flush=True)
        del net; torch.cuda.empty_cache()
    finish(arch, seeds)

In [ ]:
# hybrid_gbm: the hybrid network embedding plus the features, classified by LightGBM
if 'hybrid' in CNN_MODELS and not os.path.exists(probs_path('hybrid_gbm', 'val')):
    net = cc.load_cnn(f'{OUT}/models/hybrid_s{CNN_SEEDS["hybrid"][0]}.pt')
    def with_emb(T_, C_):
        e = cc.predict_cnn(net, T_, C_, tta=True, embed=True)
        return pd.concat([T_.reset_index(drop=True), pd.DataFrame(e, columns=[f'emb{i}' for i in range(e.shape[1])])], axis=1)
    Etr = with_emb(trC, Ctr)
    feats = cc.FEATURES + [c for c in Etr.columns if c.startswith('emb')]
    m = cc.fit_feature_model('lgbm', Etr, feats=feats, power=CLASS_POWER)
    np.save(probs_path('hybrid_gbm', 'train'), cc.predict_feature_model(m, Etr.iloc[sub]))
    np.save(f'{OUT}/probs/hybrid_gbm_train_rows.npy', sub)
    del Etr
    json.dump(list(m.labels_), open(f'{OUT}/probs/hybrid_gbm_classes.json', 'w'))
    for split, T_, C_ in (('test', Tt, Ct), ('val', Tv, Cv)):
        np.save(probs_path('hybrid_gbm', split), cc.predict_feature_model(m, with_emb(T_, C_)))
    print('hybrid_gbm done')

In [ ]:
# every model on validation with the default decision, and the CNN training curves
ALL = [n for n in ['always_aggregate'] + FEATURE_MODELS + CNN_MODELS + ['hybrid_gbm'] if os.path.exists(probs_path(n, 'val'))]
V = pd.DataFrame({n: evaluate(n)[0] for n in ALL}).T
V[COLS].to_csv(f'{OUT}/validation.csv')
show(V[COLS], SELECT_BY)
fig, ax = plt.subplots(figsize=(7, 4))
for f in sorted(glob.glob(f'{OUT}/models/*_history.csv')):
    h = pd.read_csv(f); ax.plot(h.epoch, h.val_score, label=os.path.basename(f)[:-12])
ax.set_xlabel('epoch'); ax.set_ylabel('validation balanced accuracy (no TTA)'); ax.legend(fontsize=7); plt.show()

## Part D: decisions and ensembles (validation)

In [ ]:
# grid over frames per particle, logit adjustment and ensembles, chosen on validation
V = pd.read_csv(f'{OUT}/validation.csv', index_col=0)
ALL = [n for n in V.index if n != 'always_aggregate']
GRID = [(K, h, t) for K in K_CHOICES for h in ('geo', 'mean') for t in TAU_CHOICES]
if not os.path.exists(f'{OUT}/decisions.csv'):
    grid = [dict(model=n, K=K, how=h, tau=t, **evaluate(n, 'val', K, h, t)[0]) for n in ALL for K, h, t in GRID]
    pd.DataFrame(grid).to_csv(f'{OUT}/decisions.csv', index=False)
grid = pd.read_csv(f'{OUT}/decisions.csv')
grid['K'] = grid.K.astype(object).where(grid.K.notna(), None)
best = grid.loc[grid.groupby('model')[SELECT_BY].idxmax()].set_index('model').sort_values(SELECT_BY, ascending=False)
show(best[COLS + ['K', 'how', 'tau']])

def cfg_of(row):
    return dict(K=None if row.K is None or pd.isna(row.K) else int(row.K), how=row.how, tau=float(row.tau))

order = list(best.index)
chosen, cur = [order[0]], best.loc[order[0], SELECT_BY]
ens_cfg = cfg_of(best.loc[order[0]])
for cand in order[1:]:
    if len(chosen) >= MAX_ENSEMBLE:
        break
    scores = {c: evaluate(chosen + [cand], 'val', *c)[0][SELECT_BY] for c in GRID}
    c_best = max(scores, key=scores.get)
    print(f'+ {cand}: {scores[c_best]:.3f} (current {cur:.3f})')
    if scores[c_best] > cur + 0.002:
        chosen.append(cand); cur = scores[c_best]; ens_cfg = dict(K=c_best[0], how=c_best[1], tau=float(c_best[2]))
final = dict(select_by=SELECT_BY, scheme=SCHEME, reject=REJECT, carbon_tau=CARBON_TAU, single=dict(members=[order[0]], **cfg_of(best.loc[order[0]])))
if len(chosen) > 1:
    final['ensemble'] = dict(members=chosen, **ens_cfg)
final['chosen'] = 'ensemble' if 'ensemble' in final else 'single'
json.dump(final, open(f'{OUT}/final_config.json', 'w'), indent=1)
print(json.dumps(final, indent=1))

In [ ]:
# best single-model setting under each selection metric
grid = pd.read_csv(f'{OUT}/decisions.csv')
rows = {}
for crit in ('accuracy', 'balanced_accuracy', 'acc_bal_hmean', 'carbon_weighted_accuracy', 'macro_f1'):
    b = grid.loc[grid[crit].idxmax()]
    rows[f'best {crit}'] = b
show(pd.DataFrame(rows).T[['model', 'K', 'how', 'tau'] + COLS])

## Part E: overfitting checks

In [ ]:
# training vs validation scores of every model
V = pd.read_csv(f'{OUT}/validation.csv', index_col=0)
tr = tables('train')[0]
rows = []
for n in [n for n in V.index if os.path.exists(probs_path(n, 'train'))]:
    rr = f'{OUT}/probs/{n}_train_rows.npy'
    trn = tr.iloc[np.load(rr)] if os.path.exists(rr) else tr
    trn = trn.assign(track_id=trn.pkey.factorize()[0])
    pt = cc.particle_table(trn, np.load(probs_path(n, 'train')), classes_of(n))
    true = trn.groupby(trn.seq_id + ':' + trn.track_id.astype(str)).label.first().reindex(pt.index)
    ok = pd.Series(pt.idxmax(1).values == true.values)
    rows.append(dict(model=n, train_accuracy=ok.mean(), train_balanced=ok.groupby(true.values).mean().mean(),
                     val_accuracy=V.loc[n, 'accuracy'], val_balanced=V.loc[n, 'balanced_accuracy']))
gap = pd.DataFrame(rows).set_index('model')
gap['gap_balanced'] = gap.train_balanced - gap.val_balanced
gap.to_csv(f'{OUT}/overfitting_gap.csv')
display((gap * 100).round(1).sort_values('val_balanced', ascending=False))

In [ ]:
# cross-validation over the training sequences for the best feature model
B = pd.read_csv(f'{OUT}/validation_feature_models.csv', index_col=0).drop('always_aggregate', errors='ignore')
best_feat = B[SELECT_BY].idxmax()
if tr.seq_id.nunique() < 2:
    print('cross-validation needs at least 2 training sequences')
elif not os.path.exists(f'{OUT}/overfitting_cv.csv'):
    P_oof, cl_oof = cc.oof_feature_probs(best_feat, tr, n_folds=4, power=CLASS_POWER)
    np.save(f'{OUT}/probs/{best_feat}_oof.npy', P_oof)
    json.dump(list(cl_oof), open(f'{OUT}/probs/{best_feat}_oof_classes.json', 'w'))
    pt, info = cc.train_particles(tr, P_oof, cl_oof)
    true = tr.groupby('pkey').label.first().reindex(pt.index)
    ok = pd.Series(pt.idxmax(1).values == true.values)
    pd.DataFrame([dict(model=best_feat, oof_accuracy=ok.mean(), oof_balanced=ok.groupby(true.values).mean().mean(),
                       val_accuracy=B.loc[best_feat, 'accuracy'], val_balanced=B.loc[best_feat, 'balanced_accuracy'])]) \
        .to_csv(f'{OUT}/overfitting_cv.csv', index=False)
if os.path.exists(f'{OUT}/overfitting_cv.csv'):
    display((pd.read_csv(f'{OUT}/overfitting_cv.csv').set_index('model') * 100).round(1))

In [ ]:
# learning curve: more training sequences vs validation score
if not os.path.exists(f'{OUT}/overfitting_learning_curve.csv'):
    Tv = tables('val')[0]
    tseqs = sorted(tr.seq_id.unique()); rng = np.random.default_rng(0); lc = []
    for n_s in sorted({2, 4, 8, 16, len(tseqs)}):
        if n_s > len(tseqs):
            continue
        for rep in range(2 if n_s < len(tseqs) else 1):
            pick = rng.choice(tseqs, n_s, replace=False)
            m = cc.fit_feature_model(best_feat, tr[tr.seq_id.isin(pick)], power=CLASS_POWER, seed=rep)
            r, *_ = cc.evaluate_probs(Tv, cc.predict_feature_model(m, Tv), m.labels_, None, 'geo', REJECT, 0, None,
                                      None, None, truth_arr(), SCHEME)
            lc.append(dict(n_train_sequences=n_s, rep=rep, **r))
    pd.DataFrame(lc).to_csv(f'{OUT}/overfitting_learning_curve.csv', index=False)
lc = pd.read_csv(f'{OUT}/overfitting_learning_curve.csv').groupby('n_train_sequences')[['accuracy', 'balanced_accuracy', 'carbon_weighted_accuracy']].mean()
lc.plot(marker='o'); plt.ylabel('validation score'); plt.title(f'learning curve, {best_feat}'); plt.show()
display((lc * 100).round(1))

In [ ]:
# the chosen configuration by particle size and by class (validation)
final = json.load(open(f'{OUT}/final_config.json'))
r, info, pred, cp, per = run_config(final[final['chosen']], 'val')
ta = info.true_arrival.values
d = pd.DataFrame({'true': cc.scheme_label(info.source_class.values, SCHEME)[ta], 'pred': pred.values[ta],
                  'esd': info.esd_um.values[ta], 'sprite': info.source_uid.values[ta]})
d['ok'] = d.true == d.pred
d['size'] = pd.cut(d.esd, [0, 200, 300, 500, 800, 1e6], labels=['<200', '200-300', '300-500', '500-800', '>800 um'])
display(d.groupby('size', observed=True).agg(n=('ok', 'size'), accuracy=('ok', 'mean'),
                                             share_aggregate=('true', lambda x: (x == 'aggregate').mean())).round(3))
display(d.groupby('true').agg(n=('ok', 'size'), sprites=('sprite', 'nunique'), recall=('ok', 'mean')).round(3))
display(cc.confusion(info, pred.values, SCHEME))

## Part F: carbon (validation)

In [ ]:
# carbon of the chosen configuration on validation, and the per-class carbon re-fit
final = json.load(open(f'{OUT}/final_config.json'))
c = final[final['chosen']]
r, info, pred, cp, per = run_config(c, 'val')
ta = info.true_arrival.values
sm = cc.fit_size_model(tables('train')[0])
esd_cal = cc.apply_size_model(sm, info)
ok = (info.uid > 0).values
print('median measured / true ESD:', round(float(np.median(info.esd_um.values[ok] / info.esd_true.values[ok])), 3),
      '| calibrated / true:', round(float(np.median(esd_cal[ok] / info.esd_true.values[ok])), 3))
if CARBON_SCALES == 'val':
    scales = cc.fit_class_scales(info.carbon_class.values[ta], esd_cal[ta], info.carbon_true.values[ta])
elif CARBON_SCALES == 'train_oof':
    B = pd.read_csv(f'{OUT}/validation_feature_models.csv', index_col=0).drop('always_aggregate', errors='ignore')
    bf = B[SELECT_BY].idxmax()
    assert os.path.exists(f'{OUT}/probs/{bf}_oof.npy'), 'run E2 first'
    cl_oof = json.load(open(f'{OUT}/probs/{bf}_oof_classes.json'))
    ptt, it = cc.train_particles(tables('train')[0], np.load(f'{OUT}/probs/{bf}_oof.npy'), cl_oof)
    scales = cc.fit_class_scales(cc.decide(ptt, cl_oof, REJECT).values, cc.apply_size_model(sm, it), it.carbon_true.values)
else:
    scales = {}
final['carbon'] = dict(size_model=sm, scales=scales, scales_from=CARBON_SCALES)
json.dump(final, open(f'{OUT}/final_config.json', 'w'), indent=1)
print('class carbon scales:', {k: round(v, 3) for k, v in scales.items()})
strat = {}
for name, esd, sc in (('measured size', info.esd_um.values, {}), ('calibrated size', esd_cal, {}),
                      ('calibrated size + class scales' + (' (in-sample)' if CARBON_SCALES == 'val' else ''), esd_cal, scales)):
    rr, _ = cc.score(info, pred.values, cc.particle_carbon(info.carbon_class.values, esd, sc), truth_arr(), SCHEME)
    strat[name] = {k: rr[k] for k in ('carbon_error_found', 'carbon_error_all')}
carbon_tab = pd.DataFrame(strat).T
carbon_tab.to_csv(f'{OUT}/carbon_strategies_val.csv')
display((carbon_tab * 100).round(1))
cpv = cc.particle_carbon(info.carbon_class.values, esd_cal, scales)
by = pd.DataFrame({'true': pd.Series(info.carbon_true.values[ta]).groupby(cc.scheme_label(info.source_class.values, SCHEME)[ta]).sum(),
                   'predicted': pd.Series(cpv[ta]).groupby(info.carbon_class.values[ta]).sum()}).fillna(0) * 1000
by.to_csv(f'{OUT}/carbon_by_class_val.csv')
by.plot.bar(color=['#8a8a85', '#2a78d6']); plt.ylabel('carbon (ug), found particles, validation'); plt.show()
display(by.round(3))
# where 'carbon error, all' comes from: only the 'arrival' row depends on the classifier and the size model
budget = cc.carbon_budget(info, cpv, truth_arr())
budget.to_csv(f'{OUT}/carbon_budget_val.csv')
display(budget.round(2))

## Part G: final test (scored once)

In [ ]:
# final test: the chosen configuration on the test sequences, scored once
final = json.load(open(f'{OUT}/final_config.json'))
assert 'carbon' in final, 'run Part F first (it adds the carbon calibration to final_config.json)'
res, blocks = {}, {}
for key in [k for k in ('single', 'ensemble') if k in final]:
    c = final[key]
    r, info_t, pred_t, cp_t, per_t = run_config(c, 'test', final['carbon'])
    ci = cc.bootstrap_ci(info_t, pred_t.values, cp_t, truth_arr(), SCHEME)
    label = key + ': ' + ' + '.join(c['members'])
    res[label] = {**r, **{f'{k} 95% CI': f'{100 * lo:.1f} to {100 * hi:.1f}' for k, (lo, hi) in ci.items()}}
    blocks[label] = (info_t, pred_t, cp_t, per_t)
TEST = pd.DataFrame(res).T
show(TEST)
pcs = ['accuracy', 'balanced_accuracy', 'acc_bal_hmean', 'flux_balanced_accuracy', 'macro_f1', 'carbon_weighted_accuracy', 'accuracy_esd_ge_500um']
print('scores at or above 90 %:', {lab: [k for k in pcs if float(TEST.loc[lab, k]) >= 0.9] for lab in TEST.index})
print('|carbon error| within 20 %:', {lab: [k for k in ('carbon_error_found', 'carbon_error_all') if abs(float(TEST.loc[lab, k])) <= 0.2] for lab in TEST.index})

In [ ]:
# test results per sequence, per class and the confusion matrix
label = [l for l in blocks if l.startswith(final['chosen'])][0]
info_t, pred_t, cp_t, per_t = blocks[label]
seq_tab = cc.per_sequence(info_t, pred_t.values, cp_t, truth_arr(), SCHEME).set_index('seq_id')
conf = cc.confusion(info_t, pred_t.values, SCHEME)
ALL = [n for n in pd.read_csv(f'{OUT}/validation.csv', index_col=0).index]
ref = pd.DataFrame({n: evaluate(n, 'test')[0] for n in ALL}).T
with pd.ExcelWriter(f'{OUT}/TEST_results.xlsx') as xw:
    TEST.to_excel(xw, sheet_name='final (scored once)')
    seq_tab.to_excel(xw, sheet_name='per test sequence')
    conf.to_excel(xw, sheet_name='confusion')
    cc.carbon_budget(info_t, cp_t, truth_arr()).to_excel(xw, sheet_name='carbon budget')
    per_t.to_frame('recall').to_excel(xw, sheet_name='recall by class')
    ref[COLS].to_excel(xw, sheet_name='all models (reference only)')
    pd.read_csv(f'{OUT}/validation.csv', index_col=0).to_excel(xw, sheet_name='validation, all models')
    for f, sh_ in (('overfitting_gap.csv', 'overfitting gap'), ('decisions.csv', 'decision grid (val)')):
        if os.path.exists(f'{OUT}/{f}'):
            pd.read_csv(f'{OUT}/{f}').to_excel(xw, sheet_name=sh_, index=False)
show(seq_tab[[c for c in COLS if c in seq_tab.columns]])
display(conf)
display(cc.carbon_budget(info_t, cp_t, truth_arr()).round(2))

fig, ax = plt.subplots(1, 3, figsize=(17, 5))
m = conf.div(conf.sum(1), axis=0)
ax[0].imshow(m.values, cmap='Blues', vmin=0, vmax=1)
ax[0].set_xticks(range(m.shape[1]), m.columns, rotation=60, ha='right'); ax[0].set_yticks(range(m.shape[0]), m.index)
ax[0].set_title('test confusion (rows = true class, row-normalised)')
per_t.sort_values().plot.barh(ax=ax[1], color='#2a78d6'); ax[1].set_xlim(0, 1); ax[1].set_title('test recall by true class')
ta = info_t.true_arrival.values
cb = pd.DataFrame({'true': pd.Series(info_t.carbon_true.values[ta]).groupby(cc.scheme_label(info_t.source_class.values, SCHEME)[ta]).sum(),
                   'predicted': pd.Series(cp_t[ta]).groupby(info_t.carbon_class.values[ta]).sum()}).fillna(0) * 1000
cb.plot.bar(ax=ax[2], color=['#8a8a85', '#2a78d6']); ax[2].set_ylabel('carbon (ug)'); ax[2].set_title('test carbon by class (found particles)')
plt.tight_layout(); plt.savefig(f'{OUT}/test_summary.png', dpi=150); plt.show()
print('saved', f'{OUT}/TEST_results.xlsx')

## Part H: resolution test

In [ ]:
# Experiment 1 by particle size in um and in px: where do the CNNs help?
grid = pd.read_csv(f'{OUT}/decisions.csv')
grid['K'] = grid.K.astype(object).where(grid.K.notna(), None)
best = grid.loc[grid.groupby('model')[SELECT_BY].idxmax()].set_index('model')
CNN_FAMILY = set(cc.CNN_MODELS) | {'hybrid_gbm'}
u = SC.UM_PER_PX


def bins(edges_um, labels):
    return [-np.inf] + list(edges_um) + [np.inf], labels


lab_um = [f'<{SIZE_EDGES_UM[0]} um ({SIZE_EDGES_UM[0] / u:.0f} px)'] + \
         [f'{a}-{b} um ({a / u:.0f}-{b / u:.0f} px)' for a, b in zip(SIZE_EDGES_UM[:-1], SIZE_EDGES_UM[1:])] + \
         [f'>={SIZE_EDGES_UM[-1]} um ({SIZE_EDGES_UM[-1] / u:.0f} px)']
e_px = [p * u for p in SIZE_EDGES_PX]                                  # px edges at this resolution, in um
lab_px = [f'<{SIZE_EDGES_PX[0]} px ({e_px[0]:.0f} um)'] + \
         [f'{a}-{b} px ({ea:.0f}-{eb:.0f} um)' for a, b, ea, eb in
          zip(SIZE_EDGES_PX[:-1], SIZE_EDGES_PX[1:], e_px[:-1], e_px[1:])] + \
         [f'>={SIZE_EDGES_PX[-1]} px ({e_px[-1]:.0f} um)']

rows = []
for n, b in best.iterrows():
    if n == 'always_aggregate':
        continue
    K = None if b.K is None or pd.isna(b.K) else int(b.K)
    for split in ('val', 'test'):
        r, info, pred, cp, per = evaluate(n, split, K, b.how, float(b.tau))
        ta = info.true_arrival.values
        rows.append(pd.DataFrame({'model': n, 'split': split, 'esd': info.esd_um.values[ta],
                                  'true': cc.scheme_label(info.source_class.values, SCHEME)[ta], 'pred': pred.values[ta]}))
D = pd.concat(rows, ignore_index=True)
D['ok'] = D.true == D.pred
D['family'] = np.where(D.model.isin(CNN_FAMILY), 'CNN', 'features')


def summ(g):
    rec = g.groupby('true').ok.mean()
    na = g[g.true != 'aggregate']
    return pd.Series(dict(n=len(g), accuracy=g.ok.mean(), balanced_accuracy=rec.mean(),
                          recall_not_aggregate=na.ok.mean() if len(na) else np.nan, n_not_aggregate=len(na)))


tables_h, figs = {}, []
for name, (edges, labels) in {'by um': bins(SIZE_EDGES_UM, lab_um), f'by px at {u:g} um/px': bins(e_px, lab_px)}.items():
    D['size'] = pd.cut(D.esd, edges, labels=labels, right=False)
    S_ = D.groupby(['model', 'size'], observed=True).apply(summ).astype(float)
    S_['family'] = np.where(S_.index.get_level_values('model').isin(CNN_FAMILY), 'CNN', 'features')
    fam = S_.groupby(['family', 'size'], observed=True)[['accuracy', 'balanced_accuracy', 'recall_not_aggregate',
                                                         'n_not_aggregate']].mean()   # average over the models
    bal = S_.balanced_accuracy.unstack('size').reindex(columns=labels)
    best_of = lambda B: B.apply(lambda c: c.idxmax() if c.notna().any() else None)     # NaN-safe arg-max per bin
    bc, bf = bal[bal.index.isin(CNN_FAMILY)], bal[~bal.index.isin(CNN_FAMILY)]
    cmp_ = pd.DataFrame({'best CNN': bc.max(), 'best CNN model': best_of(bc),
                         'best features': bf.max(), 'best features model': best_of(bf)}).reindex(labels)
    cmp_['CNN - features'] = cmp_['best CNN'] - cmp_['best features']
    cmp_['non-aggregate particles'] = S_.n_not_aggregate.groupby('size', observed=True).median().reindex(labels)
    tables_h[name] = (S_, fam, bal, cmp_)
    print(f'===== {name}: CNN vs feature models (averages over the models; %)')
    display((fam[['accuracy', 'balanced_accuracy', 'recall_not_aggregate']] * 100).round(1)
            .join(fam.n_not_aggregate.round(0)).unstack('family'))
    print('best CNN vs best feature model per bin (balanced accuracy, %)')
    c = cmp_.copy()
    for k in ('best CNN', 'best features', 'CNN - features'):
        c[k] = (c[k].astype(float) * 100).round(1)
    display(c)
    print('every model, balanced accuracy by size (%)')
    display((bal * 100).round(1))

with pd.ExcelWriter(f'{OUT}/resolution_test.xlsx') as xw:
    for name, (S_, fam, bal, cmp_) in tables_h.items():
        tag = 'um' if name == 'by um' else 'px'
        cmp_.to_excel(xw, sheet_name=f'best per bin ({tag})')
        fam.to_excel(xw, sheet_name=f'family averages ({tag})')
        bal.to_excel(xw, sheet_name=f'every model ({tag})')
        S_.to_excel(xw, sheet_name=f'all scores ({tag})')

# the figure: (a) family averages, (b) best CNN vs best features, for the um bins (the report's figure)
S_, fam, bal, cmp_ = tables_h['by um']
fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
x = np.arange(len(lab_um))
for k, (fm, col) in enumerate([('CNN', '#2a78d6'), ('features', '#d6862a')]):
    f_ = fam.xs(fm, level='family').reindex(lab_um)
    ax[0].plot(x, f_.balanced_accuracy * 100, 'o-', color=col, label=f'{fm}: balanced accuracy')
    ax[0].plot(x, f_.recall_not_aggregate * 100, 's--', color=col, alpha=0.7, label=f'{fm}: non-aggregate recall')
for i, n in enumerate(cmp_['non-aggregate particles']):
    ax[0].annotate(f'n={n:.0f}', (x[i], 2), ha='center', fontsize=8, color='grey')
ax[0].set_xticks(x, lab_um, rotation=15, fontsize=8); ax[0].set_ylim(0, 100); ax[0].set_ylabel('%')
ax[0].set_title(f'(a) average of the CNN-based and feature models ({u:g} um/px)', fontsize=10); ax[0].legend(fontsize=7)
w = 0.38
ax[1].bar(x - w / 2, cmp_['best CNN'].astype(float) * 100, w, color='#2a78d6', label='best CNN')
ax[1].bar(x + w / 2, cmp_['best features'].astype(float) * 100, w, color='#d6862a', label='best feature model')
for i, d in enumerate(cmp_['CNN - features'].astype(float) * 100):
    ax[1].annotate(f'{d:+.1f}', (x[i], 2), ha='center', fontsize=9, color='black')
ax[1].set_xticks(x, lab_um, rotation=15, fontsize=8); ax[1].set_ylim(0, 100); ax[1].set_ylabel('balanced accuracy %')
ax[1].set_title('(b) best CNN and best feature model per size bin (difference at the bottom)', fontsize=10)
ax[1].legend(fontsize=8)
plt.tight_layout(); plt.savefig(f'{OUT}/resolution_test.png', dpi=150); plt.show()
print('saved', f'{OUT}/resolution_test.xlsx and resolution_test.png')